# 🔬 Ablation Study
## No Post-LSTM vs No Pre-LSTM MR-ResBlock

**Run ALL cells in order (1 → 2 → 3 → 4 → 5 → 6 → 7)**

Make sure `baseline_cnn_bilstm_best.pt` is in the same folder.

In [1]:
# ============================================================
# STEP 0: Imports + Settings + Seed
# ============================================================
import os, copy, random, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# ---------- SETTINGS ----------
MANIFEST_CSV = r"C:\Users\PC\Desktop\courses\Conferences\Sevile\test\dataset\manifest_pairs.csv"
ALL_SUBJECTS = ["SUB1","SUB2","SUB3","SUB4","SUB5","SUB6","SUB7","SUB8","SUB9","SUB10"]

TARGET_CH  = 127
T_FIXED    = 1349
MEL_BINS   = 80
MEL_FRAMES = 259

BATCH_SIZE   = 4
PATIENCE     = 8
WEIGHT_DECAY = 1e-4

# Phase 1: Train baseline CNN-BiLSTM
EPOCHS_BASELINE = 60
LR_BASELINE     = 1e-4

# Phase 2: Fine-tune hybrid model
EPOCHS_HYBRID   = 60
LR_HYBRID       = 5e-5   # أقل لأننا نبدأ من checkpoint جيد

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- SEED ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f"Device: {device}")

Device: cpu


In [2]:
# ============================================================
# STEP 1: Utility Functions (Time crop/pad)
# ============================================================
def pad_or_crop_time(x, T_fixed=T_FIXED):
    C, T = x.shape
    if T == T_fixed: return x
    if T > T_fixed:  return x[:, :T_fixed]
    out = np.zeros((C, T_fixed), dtype=x.dtype)
    out[:, :T] = x
    return out

def center_crop_time(x, T_fixed=T_FIXED):
    C, T = x.shape
    if T <= T_fixed: return pad_or_crop_time(x, T_fixed)
    start = (T - T_fixed) // 2
    return x[:, start:start + T_fixed]

def random_crop_time(x, T_fixed=T_FIXED):
    C, T = x.shape
    if T <= T_fixed: return pad_or_crop_time(x, T_fixed)
    start = np.random.randint(0, T - T_fixed + 1)
    return x[:, start:start + T_fixed]

print("✅ Utility functions ready")

✅ Utility functions ready


In [3]:
# ============================================================
# STEP 2: Per-Subject EEG Statistics
# ============================================================
def compute_eeg_stats_train(manifest_csv, subject):
    df = pd.read_csv(manifest_csv)
    df = df[(df["subject"] == subject) & (df["split"] == "train")].reset_index(drop=True)
    sum_c = sumsq_c = None
    count_t = 0
    for p in df["eeg_path"].tolist():
        x = np.load(p)
        if x.shape[0] > x.shape[1]: x = x.T
        x = x.astype(np.float64)
        if sum_c is None:
            sum_c   = x.sum(axis=1)
            sumsq_c = (x**2).sum(axis=1)
        else:
            sum_c   += x.sum(axis=1)
            sumsq_c += (x**2).sum(axis=1)
        count_t += x.shape[1]
    mean = (sum_c / count_t).astype(np.float32)
    var  = (sumsq_c / count_t - mean.astype(np.float64)**2).astype(np.float32)
    std  = np.sqrt(np.maximum(var, 1e-8)).astype(np.float32)
    return mean, std

eeg_stats = {s: compute_eeg_stats_train(MANIFEST_CSV, s) for s in ALL_SUBJECTS}
print("✅ Computed stats for", len(eeg_stats), "subjects")

✅ Computed stats for 10 subjects


In [4]:
# ============================================================
# STEP 3: Dataset
# ============================================================
class MultiSubjectMaskedDataset(Dataset):
    def __init__(self, manifest_csv, split, subjects, eeg_stats):
        self.df = pd.read_csv(manifest_csv)
        self.df = self.df[(self.df["split"] == split) &
                          (self.df["subject"].isin(subjects))].reset_index(drop=True)
        self.split     = split
        self.subjects  = list(subjects)
        self.sub2idx   = {s: i for i, s in enumerate(self.subjects)}
        self.eeg_stats = eeg_stats

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        subj = row["subject"]
        eeg  = np.load(row["eeg_path"])
        if eeg.shape[0] > eeg.shape[1]: eeg = eeg.T
        eeg = eeg.astype(np.float32)
        mel = np.load(row["mel_path"]).astype(np.float32)

        mean, std = self.eeg_stats[subj]
        eeg = (eeg - mean[:, None]) / (std[:, None] + 1e-8)

        C, T = eeg.shape
        eeg_pad = np.zeros((TARGET_CH, T), dtype=np.float32)
        eeg_pad[:C, :] = eeg
        mask = np.zeros((TARGET_CH, 1), dtype=np.float32)
        mask[:C, 0] = 1.0

        if self.split == "train":
            eeg_pad = random_crop_time(eeg_pad)
        else:
            eeg_pad = center_crop_time(eeg_pad)

        sid = self.sub2idx[subj]
        return (torch.from_numpy(eeg_pad), torch.from_numpy(mel),
                torch.from_numpy(mask), torch.tensor(sid, dtype=torch.long))

print("✅ Dataset class defined")

✅ Dataset class defined


In [5]:
# ============================================================
# STEP 4: DataLoaders
# ============================================================
train_set = MultiSubjectMaskedDataset(MANIFEST_CSV, "train", ALL_SUBJECTS, eeg_stats)
val_set   = MultiSubjectMaskedDataset(MANIFEST_CSV, "val",   ALL_SUBJECTS, eeg_stats)
test_set  = MultiSubjectMaskedDataset(MANIFEST_CSV, "test",  ALL_SUBJECTS, eeg_stats)

sub_counts = train_set.df["subject"].value_counts().to_dict()
weights = [1.0 / sub_counts[s] for s in train_set.df["subject"].tolist()]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,  num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,  num_workers=0)

print(f"✅ Loaders: Train={len(train_set)}, Val={len(val_set)}, Test={len(test_set)}")

✅ Loaders: Train=796, Val=99, Test=101


In [6]:
# ============================================================
# STEP 5-B: NeuroTalk Multi-Receptive ResBlock
# ============================================================
class MultiReceptiveResBlock(nn.Module):
    """
    Core component from NeuroTalk (Lee et al., AAAI 2023).
    Parallel Conv1d with kernels 3,5,7 → merge → residual.
    Captures multi-scale frequency characteristics.
    """
    def __init__(self, channels):
        super().__init__()
        self.conv_k3 = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels), nn.LeakyReLU(0.2))
        self.conv_k5 = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(channels), nn.LeakyReLU(0.2))
        self.conv_k7 = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=7, padding=3),
            nn.BatchNorm1d(channels), nn.LeakyReLU(0.2))
        self.merge = nn.Sequential(
            nn.Conv1d(channels * 3, channels, kernel_size=1),
            nn.BatchNorm1d(channels), nn.LeakyReLU(0.2))
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        h = torch.cat([self.conv_k3(x), self.conv_k5(x), self.conv_k7(x)], dim=1)
        return x + self.dropout(self.merge(h))

print("✅ MultiReceptiveResBlock defined")

✅ MultiReceptiveResBlock defined


In [7]:
# ============================================================
# 🔬 ABLATION STUDY: Two Experiments
# ============================================================
# A: No Post-LSTM MR-ResBlock (4 blocks: conv1,conv2,conv3,pre-LSTM)
# B: No Pre-LSTM MR-ResBlock (4 blocks: conv1,conv2,conv3,post-LSTM)
# ============================================================

# ============================================================
# ABLATION A: No Post-LSTM (4 MR-ResBlocks)
# ============================================================
class HybridNeuroTalk_NoPostLSTM(nn.Module):
    """Remove post-LSTM MR-ResBlock. Keeps: mr_res1,2,3 + pre-LSTM (4 blocks)"""
    def __init__(self, n_subjects=10, embed_dim=32, hidden=256):
        super().__init__()
        self.sub_embed = nn.Embedding(n_subjects, embed_dim)
        self.conv1 = nn.Sequential(
            nn.Conv1d(TARGET_CH, 128, kernel_size=7, padding=3),
            nn.ReLU(), nn.MaxPool1d(2))
        self.conv2 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=7, padding=3),
            nn.ReLU(), nn.MaxPool1d(2))
        self.conv3 = nn.Sequential(
            nn.Conv1d(256, 256, kernel_size=7, padding=3),
            nn.ReLU(), nn.MaxPool1d(2))
        self.mr_res1 = MultiReceptiveResBlock(128)
        self.mr_res2 = MultiReceptiveResBlock(256)
        self.mr_res3 = MultiReceptiveResBlock(256)
        self.mr_pre_lstm = MultiReceptiveResBlock(256)
        self.time_pool = nn.AdaptiveAvgPool1d(MEL_FRAMES)
        self.bilstm = nn.LSTM(input_size=256+embed_dim, hidden_size=hidden,
                              num_layers=1, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(hidden * 2, MEL_BINS)  # Direct 512->80

    def forward(self, x, mask, sid):
        x = x * mask
        z = self.conv1(x);  z = self.mr_res1(z)
        z = self.conv2(z);  z = self.mr_res2(z)
        z = self.conv3(z);  z = self.mr_res3(z)
        z = self.mr_pre_lstm(z)
        z = self.time_pool(z).permute(0, 2, 1)
        e = self.sub_embed(sid).unsqueeze(1).repeat(1, z.size(1), 1)
        z = torch.cat([z, e], dim=-1)
        z, _ = self.bilstm(z)
        z = self.proj(z).permute(0, 2, 1)
        return z

# ============================================================
# ABLATION B: No Pre-LSTM (4 MR-ResBlocks)
# ============================================================
class HybridNeuroTalk_NoPreLSTM(nn.Module):
    """Remove pre-LSTM MR-ResBlock. Keeps: mr_res1,2,3 + post-LSTM (4 blocks)"""
    def __init__(self, n_subjects=10, embed_dim=32, hidden=256):
        super().__init__()
        self.sub_embed = nn.Embedding(n_subjects, embed_dim)
        self.conv1 = nn.Sequential(
            nn.Conv1d(TARGET_CH, 128, kernel_size=7, padding=3),
            nn.ReLU(), nn.MaxPool1d(2))
        self.conv2 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=7, padding=3),
            nn.ReLU(), nn.MaxPool1d(2))
        self.conv3 = nn.Sequential(
            nn.Conv1d(256, 256, kernel_size=7, padding=3),
            nn.ReLU(), nn.MaxPool1d(2))
        self.mr_res1 = MultiReceptiveResBlock(128)
        self.mr_res2 = MultiReceptiveResBlock(256)
        self.mr_res3 = MultiReceptiveResBlock(256)
        self.time_pool = nn.AdaptiveAvgPool1d(MEL_FRAMES)
        self.bilstm = nn.LSTM(input_size=256+embed_dim, hidden_size=hidden,
                              num_layers=1, batch_first=True, bidirectional=True)
        self.post_lstm_proj = nn.Linear(hidden * 2, 256)
        self.mr_post_lstm = MultiReceptiveResBlock(256)
        self.proj = nn.Linear(256, MEL_BINS)

    def forward(self, x, mask, sid):
        x = x * mask
        z = self.conv1(x);  z = self.mr_res1(z)
        z = self.conv2(z);  z = self.mr_res2(z)
        z = self.conv3(z);  z = self.mr_res3(z)
        z = self.time_pool(z).permute(0, 2, 1)
        e = self.sub_embed(sid).unsqueeze(1).repeat(1, z.size(1), 1)
        z = torch.cat([z, e], dim=-1)
        z, _ = self.bilstm(z)
        z = self.post_lstm_proj(z).permute(0, 2, 1)
        z = self.mr_post_lstm(z).permute(0, 2, 1)
        z = self.proj(z).permute(0, 2, 1)
        return z

# ============================================================
# HELPER FUNCTIONS
# ============================================================
criterion = nn.MSELoss()

def run_epoch_abl(model, loader, optimizer=None, train=True):
    model.train() if train else model.eval()
    total, n = 0.0, 0
    for xb, yb, mask, sid in loader:
        xb, yb = xb.to(device), yb.to(device)
        mask, sid = mask.to(device), sid.to(device)
        yp = model(xb, mask, sid)
        loss = criterion(yp, yb)
        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total += loss.item() * xb.size(0)
        n += xb.size(0)
    return total / max(n, 1)

def transfer_weights_abl(model, name):
    baseline_sd = torch.load(CKPT_BASELINE, map_location=device)
    mapping = {
        "sub_embed.weight": "sub_embed.weight",
        "encoder.0.weight": "conv1.0.weight", "encoder.0.bias": "conv1.0.bias",
        "encoder.3.weight": "conv2.0.weight", "encoder.3.bias": "conv2.0.bias",
        "encoder.6.weight": "conv3.0.weight", "encoder.6.bias": "conv3.0.bias",
        "bilstm.weight_ih_l0": "bilstm.weight_ih_l0",
        "bilstm.weight_hh_l0": "bilstm.weight_hh_l0",
        "bilstm.bias_ih_l0": "bilstm.bias_ih_l0",
        "bilstm.bias_hh_l0": "bilstm.bias_hh_l0",
        "bilstm.weight_ih_l0_reverse": "bilstm.weight_ih_l0_reverse",
        "bilstm.weight_hh_l0_reverse": "bilstm.weight_hh_l0_reverse",
        "bilstm.bias_ih_l0_reverse": "bilstm.bias_ih_l0_reverse",
        "bilstm.bias_hh_l0_reverse": "bilstm.bias_hh_l0_reverse",
    }
    sd = model.state_dict()
    t = 0
    for sk, dk in mapping.items():
        if sk in baseline_sd and dk in sd:
            if baseline_sd[sk].shape == sd[dk].shape:
                sd[dk] = baseline_sd[sk]; t += 1
    model.load_state_dict(sd)
    print(f"✅ [{name}] Transferred {t}/{len(mapping)} weights")
    return model

def train_ablation(model, ckpt, name):
    transferred_p, new_p = [], []
    for n, p in model.named_parameters():
        if any(k in n for k in ['mr_res','mr_pre','mr_post','post_lstm_proj']):
            new_p.append(p)
        else:
            transferred_p.append(p)
    opt = torch.optim.Adam([
        {"params": transferred_p, "lr": LR_HYBRID * 0.1},
        {"params": new_p, "lr": LR_HYBRID},
    ], weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
    print(f"\n{'='*60}\nTRAINING: {name}\n{'='*60}")
    best_val, best_ep, pat = float("inf"), -1, 0
    for epoch in range(1, EPOCHS_HYBRID + 1):
        tr = run_epoch_abl(model, train_loader, opt, train=True)
        va = run_epoch_abl(model, val_loader, train=False)
        sch.step(va)
        print(f"Epoch {epoch:02d} | Train {tr:.4f} | Val {va:.4f}")
        if va < best_val:
            best_val, best_ep, pat = va, epoch, 0
            torch.save(model.state_dict(), ckpt)
            print(f"  ✅ Saved")
        else:
            pat += 1
            if pat >= PATIENCE:
                print(f"  🛑 Early stop"); break
    print(f"✅ {name} done. Best epoch={best_ep}, Val={best_val:.4f}")
    model.load_state_dict(torch.load(ckpt, map_location=device))
    return model

def evaluate_abl(model, loader, name):
    model.eval()
    from collections import defaultdict
    results = defaultdict(lambda: {"p":[],"t":[]})
    all_p, all_t = [], []
    with torch.no_grad():
        for xb, yb, mask, sid in loader:
            xb,yb = xb.to(device),yb.to(device)
            mask,sid = mask.to(device),sid.to(device)
            yp = model(xb, mask, sid)
            for i in range(xb.size(0)):
                s = ALL_SUBJECTS[sid[i].item()]
                pf = yp[i].cpu().numpy().flatten()
                tf = yb[i].cpu().numpy().flatten()
                results[s]["p"].append(pf); results[s]["t"].append(tf)
                all_p.append(pf); all_t.append(tf)
    ap, at = np.concatenate(all_p), np.concatenate(all_t)
    gpcc = np.corrcoef(ap, at)[0,1]
    gmse = np.mean((ap-at)**2)
    print(f"\n{'='*50}\n{name}: PCC={gpcc:.4f}, MSE={gmse:.4f}\n{'='*50}")
    spccs = {}
    for s in ALL_SUBJECTS:
        if s in results:
            sp,st = np.concatenate(results[s]["p"]), np.concatenate(results[s]["t"])
            spccs[s] = np.corrcoef(sp,st)[0,1]
            print(f"  {s}: PCC={spccs[s]:.3f}")
    return gpcc, gmse, spccs

# ============================================================
# RUN EXPERIMENT A: No Post-LSTM
# ============================================================
CKPT_BASELINE = r"C:/Users/PC/Desktop/courses/Conferences/Sevile/ARTICLE-TRANSFER LEARNING/baseline_cnn_bilstm_best.pt"

print("\n" + "🔬"*20 + "\nABLATION A: No Post-LSTM (4 blocks)\n" + "🔬"*20)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
model_A = HybridNeuroTalk_NoPostLSTM(n_subjects=len(ALL_SUBJECTS)).to(device)
model_A = transfer_weights_abl(model_A, "No Post-LSTM")
model_A = train_ablation(model_A, "ablation_no_post_lstm.pt", "No Post-LSTM")
pcc_A, mse_A, subj_A = evaluate_abl(model_A, test_loader, "No Post-LSTM")

# ============================================================
# RUN EXPERIMENT B: No Pre-LSTM
# ============================================================
print("\n" + "🔬"*20 + "\nABLATION B: No Pre-LSTM (4 blocks)\n" + "🔬"*20)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
model_B = HybridNeuroTalk_NoPreLSTM(n_subjects=len(ALL_SUBJECTS)).to(device)
model_B = transfer_weights_abl(model_B, "No Pre-LSTM")
model_B = train_ablation(model_B, "ablation_no_pre_lstm.pt", "No Pre-LSTM")
pcc_B, mse_B, subj_B = evaluate_abl(model_B, test_loader, "No Pre-LSTM")

# ============================================================
# FINAL SUMMARY TABLE
# ============================================================
print("\n" + "="*70)
print("📊 ABLATION STUDY — FINAL SUMMARY")
print("="*70)
print(f"\n{'Model':<35} {'PCC':>8} {'MSE':>8} {'Blocks':>8}")
print("-"*62)
print(f"{'Baseline CNN-BiLSTM':<35} {'0.2878':>8} {'4.5667':>8} {'0':>8}")
print(f"{'No Post-LSTM (4 blocks)':<35} {pcc_A:>8.4f} {mse_A:>8.4f} {'4':>8}")
print(f"{'No Pre-LSTM (4 blocks)':<35} {pcc_B:>8.4f} {mse_B:>8.4f} {'4':>8}")
print(f"{'Full Hybrid (5 blocks)':<35} {'0.3998':>8} {'4.1936':>8} {'5':>8}")
print("-"*62)
print("\n✅ DONE! Screenshot this table and send it back.")



🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬
ABLATION A: No Post-LSTM (4 blocks)
🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬
✅ [No Post-LSTM] Transferred 15/15 weights

TRAINING: No Post-LSTM
Epoch 01 | Train 78.6893 | Val 74.4899
  ✅ Saved
Epoch 02 | Train 70.5557 | Val 66.2953
  ✅ Saved
Epoch 03 | Train 62.4033 | Val 58.6220
  ✅ Saved
Epoch 04 | Train 54.8906 | Val 51.4660
  ✅ Saved
Epoch 05 | Train 48.0891 | Val 44.8275
  ✅ Saved
Epoch 06 | Train 41.8394 | Val 38.7049
  ✅ Saved
Epoch 07 | Train 35.8900 | Val 33.0983
  ✅ Saved
Epoch 08 | Train 30.4423 | Val 28.0072
  ✅ Saved
Epoch 09 | Train 25.5210 | Val 23.4293
  ✅ Saved
Epoch 10 | Train 21.5479 | Val 19.3598
  ✅ Saved
Epoch 11 | Train 17.6550 | Val 15.8003
  ✅ Saved
Epoch 12 | Train 14.3866 | Val 12.7301
  ✅ Saved
Epoch 13 | Train 11.4957 | Val 10.1772
  ✅ Saved
Epoch 14 | Train 9.1591 | Val 8.3232
  ✅ Saved
Epoch 15 | Train 7.5007 | Val 6.5653
  ✅ Saved
Epoch 16 | Train 6.0995 | Val 5.4750
  ✅ Saved
Epoch 17 | Train 5.3512 | Val 4.8291
  ✅ Saved
Epoch 18 | Train 4.8756 | Val